In [25]:
import pandas as pd
import json

In [26]:
original_path = "../data/external/all_examples_og_prompt_with_position_info_and_success_V2.json"
scale_folder = "../outputs/all_scale"

In [27]:
from collections import Counter

def any_repeated_substring(s, min_repeats=5, max_sub_len=10):
    n = len(s)
    L = max_sub_len
    counts = Counter(s[i:i+L] for i in range(n - L + 1))
    if any(v >= min_repeats for v in counts.values()):
        # print("The max repeating subsequence is - ",counts.most_common(1)[0][0])
        return True
    return False


def _downgrade_plan_to_degenerate(steered_text, original_ym = None):
    """
        - empty string, single character, single string, then classify sa degen.
        - multiple number of 'import' statements in the output without any return statement at the end.
    """
    ## if original_ym is part of the steered text, it's marked as degenerate to prevent a False Positive.
    if original_ym != None and original_ym in steered_text:
        return True

    stripped_text = steered_text.strip()
    if stripped_text == "" or (" " not in stripped_text and len(stripped_text) >= 10):
        return True
    if any_repeated_substring(steered_text) and (original_ym == "return" or (original_ym != "return" and "return" not in steered_text)):
        return True
    
    return False

In [28]:
def _classify_as_planning_vs_not_planning(steering_results, planning_analysis):
    """
        Logic:
            - if already marked as 'Not planning', we keep as-is.
            - if marked as planning, we check for non-degeneracy.
            - if marked as can't say, we check for non-degenaracy and whether the original and new normalized sequences differ by much.
        
        Returns, new planning analysis, with 
        {
            "y_m": {
                "original_verdict": 
                "new_verdict":
             }
            ....
        }
    """
    ym_keys = list(planning_analysis.keys())
    new_planning_analysis = {}
    for y_m in ym_keys:
        new_verdict = planning_analysis[y_m]
        base_suffix = steering_results[y_m]["base_text"]
         
        # if y_m == "return":
        #     new_verdict = "Not planning"
        if planning_analysis[y_m] == "Plan":  
            if not base_suffix.startswith(y_m) and not all(_downgrade_plan_to_degenerate(e.get("decoded_text", ""), y_m) for e in steering_results[y_m]["steered"]):
                new_verdict = "Plan"
            else:
                new_verdict = "Can't say"                
        
        new_planning_analysis[y_m] = {
            "original_verdict": planning_analysis[y_m],
            "new_verdict": new_verdict
        }
    
    return new_planning_analysis

In [29]:
import glob
all_tokens = [*glob.glob(scale_folder + "/base/*/*"), *glob.glob(scale_folder + "/instruct/*/*")]

In [30]:
len(all_tokens), all_tokens[0]

(1997, '../outputs/all_scale/base/prompt_0/token_188')

In [31]:
def dump_json(filename, my_dict):
    with open(filename, "w") as f:
        json.dump(my_dict, f, indent=4)  # indent=4 makes it pretty-printed

def load_json(file):
    with open(file, "r") as f:
        return json.load(f)

In [32]:
import os

for token_planning_folder in all_tokens:
    # json_files = glob.glob(f"{token_planning_folder}/*.json")
    # metadata, planning_analysis, steering_results = load_json(json_files[1]), load_json(json_files[2]), load_json(json_files[3])
    planning_analysis = load_json(f"{token_planning_folder}/planning_analysis.json")
    steering_results = load_json(f"{token_planning_folder}/steering_results.json")
    new_planning_analysis = _classify_as_planning_vs_not_planning(steering_results, planning_analysis)
    updated_folder = token_planning_folder.replace("all_scale", "t_all_scale")
    os.makedirs(updated_folder, exist_ok=True)
    dump_json(updated_folder + "/updated_planning_analysis.json", new_planning_analysis)

In [33]:
def _detect_planning(planning_dict):
    keys = []
    for key, value in planning_dict.items():
        if value["new_verdict"] == "Plan":
            keys.append(key)
    return keys

def _detect_cant_says(planning_dict):
    present = False
    for key, value in planning_dict.items():
        if value["new_verdict"] == "Can't say":
            present = True
    return present

def _get_ym_plans(folder):
    token_planning_files = glob.glob(folder + "/*/*.json")
    planning_datas = [load_json(f) for f in token_planning_files]
    y_ms = []
    for data in planning_datas:
        y_ms.extend(_detect_planning(data))
    y_ms = list(set(y_ms))
    return y_ms

def _detect_cs(folder):
    token_planning_files = glob.glob(folder + "/*/*.json")
    planning_datas = [load_json(f) for f in token_planning_files]
    for data in planning_datas:
        if _detect_cant_says(data):
            return True
    return False

def _get_base_and_instruct_plans(iter, root_folder = "../outputs/t_all_scale"):
    base_folder = f"{root_folder}/base/prompt_{iter}"
    instruct_folder = f"{root_folder}/instruct/prompt_{iter}"
    base_yms = None
    instruct_yms = None
    if os.path.exists(base_folder):
        base_yms = _get_ym_plans(base_folder)
    if os.path.exists(instruct_folder):
        instruct_yms = _get_ym_plans(instruct_folder)
    return {
        "base": base_yms,
        "instruct": instruct_yms
    }

def _get_base_and_instruct_cantsays(iter, root_folder = "../outputs/t_all_scale"):
    base_folder = f"{root_folder}/base/prompt_{iter}"
    instruct_folder = f"{root_folder}/instruct/prompt_{iter}"
    if os.path.exists(base_folder):
        base_cs = _detect_cs(base_folder)
    else:
        base_cs = False
    if os.path.exists(instruct_folder):
        instruct_cs = _detect_cs(instruct_folder)
    else:
        instruct_cs = False
    return {
        "base": base_cs,
        "instruct": instruct_cs
    }

In [34]:
data = load_json("../data/external/all_examples_og_prompt_with_position_info_and_success_V2.json")

In [35]:
for iter, entry in enumerate(data):
    ym_plans = _get_base_and_instruct_plans(iter)
    entry["base_plans"] = ym_plans["base"]
    entry["instruct_plans"] = ym_plans["instruct"]
    cs = _get_base_and_instruct_cantsays(iter)    
    entry["base_cs"] = cs["base"]
    entry["instruct_cs"] = cs["instruct"]

In [66]:
def _get_4x4_grid(selected_cases, with_e = False):
    suffix = ""
    if with_e:
        suffix = "_e"
    instruct_plan_base_no_plan = []
    base_plan_instruct_no_plan = []
    both_plans = []
    no_plans = []
    for entry in selected_cases:
        base_planning, instruct_planning = False, False
        if entry["base_plans" + suffix] and len(entry["base_plans" + suffix]) > 0:
            base_planning = True
        elif entry["base_cs" + suffix] == True:
            continue ## skip case if can't say.    
        if entry["instruct_plans" + suffix] and len(entry["instruct_plans" + suffix]) > 0:
            instruct_planning = True
        elif entry["instruct_cs" + suffix] == True:
            continue ## similar skip.

        if base_planning:
            if instruct_planning:         
                both_plans.append(entry)
            else:
                base_plan_instruct_no_plan.append(entry)
        elif instruct_planning:
            instruct_plan_base_no_plan.append(entry)
        else:
            no_plans.append(entry)
    
    print({
        "Both": len(both_plans),
        "Only instruct": len(instruct_plan_base_no_plan),
        "Only base": len(base_plan_instruct_no_plan),
        "None": len(no_plans)
    })

    return {
        "Both": both_plans,
        "Only instruct": instruct_plan_base_no_plan, 
        "Only base": base_plan_instruct_no_plan,
        "None": no_plans 
    }

In [37]:
for idx, entry in enumerate(data):
    entry["index"] = idx

with open("../data/base_vs_instruct_oracle_V1.json", "w") as f:
    json.dump(data, f, indent = 2)

In [67]:
selected_cases = [case for case in data if case["instruct_pass"] == True and case["base_pass"] == False]
len(selected_cases)

85

In [68]:
result = _get_4x4_grid(selected_cases)

{'Both': 32, 'Only instruct': 7, 'Only base': 12, 'None': 7}


In [40]:
result['None']

[{'source_file': 'Benchmark Questions Verification V2.ipynb',
  'task_id': 251,
  'prompt': 'Write a function that takes in a list and an element and inserts the element before each element in the list, and returns the resulting list.',
  'code': 'def insert_element(list,element):\n list = [v for elt in list for v in (element, elt)]\n return list',
  'test_imports': [],
  'test_list': ["assert insert_element(['Red', 'Green', 'Black'] ,'c')==['c', 'Red', 'c', 'Green', 'c', 'Black']",
   "assert insert_element(['python', 'java'] ,'program')==['program', 'python', 'program', 'java']",
   "assert insert_element(['happy', 'sad'] ,'laugh')==['laugh', 'happy', 'laugh', 'sad']"],
  'instruct_code': 'def insert_element(list1, element):\n    new_list = []\n    for i in range(len(list1)):\n        new_list.append(element)\n        new_list.append(list1[i])\n    return new_list\n',
  'model_output': 'def insert_element(list, element):\n    new_list = []\n    for i in range(len(list)):\n        if 

In [41]:
selected_cases = [case for case in data if case["instruct_pass"] == False and case["base_pass"] == True]
len(selected_cases)

10

In [42]:
result = _get_4x4_grid(selected_cases)

{'Both': 4, 'Only instruct': 0, 'Only base': 3, 'None': 0}


In [43]:
selected_cases = [case for case in data if case["instruct_pass"] == False and case["base_pass"] == False]
len(selected_cases)

197

In [44]:
result = _get_4x4_grid(selected_cases)

{'Both': 91, 'Only instruct': 12, 'Only base': 25, 'None': 18}


In [45]:
selected_cases = [case for case in data if case["instruct_pass"] == True and case["base_pass"] == True]
len(selected_cases)

100

In [46]:
result = _get_4x4_grid(selected_cases)

{'Both': 27, 'Only instruct': 6, 'Only base': 12, 'None': 13}


### By earliest position logic.

In [47]:
def _classify_earliest_position_as_planning(steering_results, planning_analysis):
    """
        Logic : 
            If the output is degenerate, we downgrade the Planning to Can't Say.

        Returns, similar new_verdict with 
        {
            "Key": {
                "original_verdict": "",
                "new_verdict": "",
            }
        }
    """
    ym_keys = list(planning_analysis.keys())
    new_planning_analysis = {}
    for y_m in ym_keys:
        new_verdict = planning_analysis[y_m]["final_label"]
        base_suffix = steering_results[y_m]["base_text"]        
        if planning_analysis[y_m]["final_label"] == "Plan":  
            if not base_suffix.startswith(y_m) and not all(_downgrade_plan_to_degenerate(e.get("decoded_text", ""), y_m) for e in steering_results[y_m]["steered"]):
                new_verdict = "Plan"
            else:
                new_verdict = "Can't say"                
        
        new_planning_analysis[y_m] = {
            "original_verdict": planning_analysis[y_m]["final_label"],
            "new_verdict": new_verdict
        }
    
    return new_planning_analysis

for token_planning_folder in all_tokens:
    metadata = load_json(f"{token_planning_folder}/metadata.json")
    new_planning_analysis = {} ## no plans anyway.
    if metadata["earliest_position"] is not None:
        planning_analysis = load_json(f"{token_planning_folder}/earliest_position_planning_analysis.json")
        steering_results = load_json(f"{token_planning_folder}/earliest_position.json")
        new_planning_analysis = _classify_earliest_position_as_planning(steering_results, planning_analysis)
    updated_folder = token_planning_folder.replace("all_scale", "t_all_scale_e")
    os.makedirs(updated_folder, exist_ok=True)
    dump_json(updated_folder + "/updated_planning_analysis.json", new_planning_analysis)

In [48]:
for iter, entry in enumerate(data):
    ym_plans = _get_base_and_instruct_plans(iter, root_folder = "../outputs/t_all_scale_e")
    entry["base_plans_e"] = ym_plans["base"]
    entry["instruct_plans_e"] = ym_plans["instruct"]
    cs = _get_base_and_instruct_cantsays(iter, root_folder = "../outputs/t_all_scale_e")    
    entry["base_cs_e"] = cs["base"]
    entry["instruct_cs_e"] = cs["instruct"]

In [56]:
selected_cases = [entry for entry in data if entry["base_pass"] == False and entry["instruct_pass"] == True]
len(selected_cases)

85

In [57]:
result = _get_4x4_grid(selected_cases, with_e = True)

{'Both': 55, 'Only instruct': 9, 'Only base': 4, 'None': 3}


In [62]:
selected_cases = [entry for entry in data if entry["base_pass"] == True and entry["instruct_pass"] == True]
len(selected_cases)

100

In [63]:
result = _get_4x4_grid(selected_cases, with_e = True)

{'Both': 63, 'Only instruct': 12, 'Only base': 5, 'None': 5}


In [64]:
selected_cases = [entry for entry in data if entry["base_pass"] == False and entry["instruct_pass"] == False]
len(selected_cases)

197

In [65]:
result = _get_4x4_grid(selected_cases, with_e = True)

{'Both': 115, 'Only instruct': 22, 'Only base': 20, 'None': 8}


In [60]:
# result['None']

In [69]:
with open("../data/base_vs_instruct_oracle_V2.json", "w") as f:
    json.dump(data, f, indent = 2)